***Note:*** If you are using Colab, uncomment the following lines and execute them. Once the installation is complete, proceed with the rest of the code.

In [2]:
# !pip install -U huggingface_hub
# ! pip install -U bitsandbytes==0.46.1

# ! pip install -U transformers==4.39.3 accelerate

# ! pip uninstall -y transformers accelerate tokenizers sentence-transformers
# ! pip uninstall -y transformers accelerate tokenizers sentence-transformers


# ! pip install \
#   transformers==4.40.2 \
#   accelerate==0.30.1 \
#   tokenizers==0.19.1 \
#   sentence-transformers==2.6.1


# ! pip install faiss-cpu
# ! pip install pypdf

Upload the ZIP folder that contains the entire application.

In [3]:
from google.colab import files
_ = files.upload()

Saving customer_support_ai.zip to customer_support_ai.zip


In [4]:
# execute this command to unzip the folder
!unzip customer_support_ai.zip

Archive:  customer_support_ai.zip
   creating: customer_support_ai/
   creating: customer_support_ai/.ipynb_checkpoints/
  inflating: customer_support_ai/.ipynb_checkpoints/unit testing-checkpoint.ipynb  
  inflating: customer_support_ai/.ipynb_checkpoints/vector_database_creation-checkpoint.ipynb  
   creating: customer_support_ai/apis/
  inflating: customer_support_ai/apis/billing_api.py  
  inflating: customer_support_ai/apis/payment_api.py  
  inflating: customer_support_ai/apis/supabase_client.py  
  inflating: customer_support_ai/apis/__init__.py  
   creating: customer_support_ai/classifiers/
  inflating: customer_support_ai/classifiers/data_required.py  
  inflating: customer_support_ai/classifiers/intent.py  
  inflating: customer_support_ai/classifiers/issue.py  
  inflating: customer_support_ai/classifiers/risk.py  
  inflating: customer_support_ai/classifiers/__init__.py  
   creating: customer_support_ai/classifiers/__pycache__/
  inflating: customer_support_ai/classifiers

In [5]:
# Add the project root directory to the Python path so modules can be imported
import sys
sys.path.append("/content/customer_support_ai")

Env set up

In [7]:
import os

# Hugging Face (required for LLM access)
os.environ["HUGGINGFACE_HUB_TOKEN"] = "<your_huggingface_token_here>"

# Supabase (API layer)
os.environ["SUPABASE_URL"] = "<your_supabase_project_url>"
os.environ["SUPABASE_KEY"] = "<your_supabase_anon_or_service_key>"

#### Start of the Application

* We will first import the required modules from the application.
* Next, we will load the main LLM (MistralAI) and the embedding model.
* Once both models are loaded globally, we will load the FAISS index and the index-to-document mapping object from the local `/vector_db` directory.
* After the application is fully initialized, we will pass sample customer queries through the system and observe the results.

In [8]:
# importing important modules
from pipeline import process_query
from vector_db import load_vector_db
from models.embeddings import load_embedding_model
from models.loader import load_mistralai

In [9]:
# loading mistralai 7B model (LLM)
tokenizer, model = load_mistralai()
print("LLM loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Mistral 7B loaded successfully (slow tokenizer).
LLM loaded.


In [10]:
# loading embedding model
embedding_model = load_embedding_model(device="cpu")
print("Embedding model loaded.")

Embedding model loaded.


In [14]:
# use this to upload the faiss index and id_to_index object, which is output of vector databse module
# ajdust the folder path as per the execution style, if root is not outside customer_support_ai then use ./vector_db/output
os.listdir("./customer_support_ai/vector_db/output")

['axis_cc_terms.index', 'idx_to_data.pkl']

In [15]:
# load the vector databse
INDEX_PATH = "./customer_support_ai/vector_db/output/axis_cc_terms.index"
DATA_PATH = "./customer_support_ai/vector_db/output/idx_to_data.pkl"

index, rag_documents = load_vector_db(
    index_path=INDEX_PATH,
    data_path=DATA_PATH
)

print("Vector DB loaded.")

Vector DB loaded.


In [16]:
# helper function to prettify the output
import json

def show_result(result):
    print("\nDECISION:")
    print(json.dumps(result["decision"], indent=2))
    print("\nRESPONSE:")
    print(result["response"])
    if "stability_score" in result:
        print("\nSTABILITY SCORE:", round(result["stability_score"], 2))

### Passing sample queries


1. **Example 1:**
   This is Test Case 1, where the customer query is a complaint. The system will call the relevant API, fetch the required information, and generate a response to address the complaint.

2. **Example 2:**
   This is Test Case 2, which triggers a second API call.
   The APIs used here contain only sample data with a single row for demonstration purposes. You can replace these functions in the module with real or different APIs as needed. This represents the action layer of the architecture, where decisions are executed after the reasoning step.

3. **Example 3:**
   This is Test Case 3, a FAQ-based question. The system will answer it using our RAG pipeline, which leverages the Terms and Conditions document to generate an accurate response.

4. **Example 4:**
   This is Test Case 4, demonstrating the escalation (human-in-the-loop) part of the architecture.
   In this case, high-risk complaints are routed to the appropriate team for manual resolution due to their sensitive nature.


In [17]:
# Example 1
account_id =  'AXIS_CC_1009'
customer_message = 'why i got late charges on my credit card'

result_1 = process_query(
    query=customer_message,
    account_id=account_id,
    index=index,
    embedding_model=embedding_model,
    rag_documents=rag_documents,
)

print(result_1)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


{'decision': {'query': 'why i got late charges on my credit card', 'intent': 'complaint', 'issue': 'billing_issue', 'risk_level': 'low', 'data_dependency': 'yes', 'answerability': 'api'}, 'response': '(ONLY CUSTOMER-FACING RESPONSE, NO HEADINGS):\n   Based on the information provided, your credit card payment was received after the due date, which was on 2025-01-24. As a result, late charges may have been applied according to our credit card policy. If you have any questions or need further assistance, please contact us.', 'stability_score': 1.0}


In [18]:
show_result(result_1)


DECISION:
{
  "query": "why i got late charges on my credit card",
  "intent": "complaint",
  "issue": "billing_issue",
  "risk_level": "low",
  "data_dependency": "yes",
  "answerability": "api"
}

RESPONSE:
(ONLY CUSTOMER-FACING RESPONSE, NO HEADINGS):
   Based on the information provided, your credit card payment was received after the due date, which was on 2025-01-24. As a result, late charges may have been applied according to our credit card policy. If you have any questions or need further assistance, please contact us.

STABILITY SCORE: 1.0


In [19]:
# Example 2
account_id = "AXIS_CC_1002"
customer_message = "My credit card payment is getting declined even though I have sufficient balance."

result_2 = process_query(
    query=customer_message,
    account_id=account_id,
    index=index,
    embedding_model=embedding_model,
    rag_documents=rag_documents,
)

print(result_2)

{'decision': {'query': 'My credit card payment is getting declined even though I have sufficient balance.', 'intent': 'complaint', 'issue': 'payment_issue', 'risk_level': 'low', 'data_dependency': 'yes', 'answerability': 'api'}, 'response': '(ONLY CUSTOMER-FACING RESPONSE, NO HEADINGS):\nYour credit card payment for the statement dated 2025-01-04 did not complete successfully. Despite having a sufficient balance on that date, the payment was not processed. If your payment is now late, additional fees may apply according to our policy. We apologize for any inconvenience this may have caused and recommend checking your payment information for any errors. If you continue to experience issues, please contact us for further assistance.', 'stability_score': 1.0}


In [20]:
show_result(result_2)


DECISION:
{
  "query": "My credit card payment is getting declined even though I have sufficient balance.",
  "intent": "complaint",
  "issue": "payment_issue",
  "risk_level": "low",
  "data_dependency": "yes",
  "answerability": "api"
}

RESPONSE:
(ONLY CUSTOMER-FACING RESPONSE, NO HEADINGS):
Your credit card payment for the statement dated 2025-01-04 did not complete successfully. Despite having a sufficient balance on that date, the payment was not processed. If your payment is now late, additional fees may apply according to our policy. We apologize for any inconvenience this may have caused and recommend checking your payment information for any errors. If you continue to experience issues, please contact us for further assistance.

STABILITY SCORE: 1.0


In [21]:
# Example 3
account_id = "AXIS_CC_1003"
query = "What are the consequences of missing a credit card payment due date?"

result_3 = process_query(
    query=query,
    account_id=account_id,
    index=index,
    embedding_model=embedding_model,
    rag_documents=rag_documents,
)

print(result_3)

{'decision': {'query': 'What are the consequences of missing a credit card payment due date?', 'intent': 'inquiry', 'issue': 'payment_issue', 'risk_level': 'low', 'data_dependency': 'no', 'answerability': 'faq'}, 'response': '1. A late payment fee, as mentioned in the Schedule of Charges, will be levied to the Card Account.\n    2. The Bank is entitled to recover the outstanding dues in accordance with relevant laws.\n    3. The Bank may appoint a third party for collection, at your cost and risk.\n    4. You will be liable for all costs associated with the collection of dues and legal expenses with interest.', 'stability_score': 1.0}


In [22]:
show_result(result_3)


DECISION:
{
  "query": "What are the consequences of missing a credit card payment due date?",
  "intent": "inquiry",
  "issue": "payment_issue",
  "risk_level": "low",
  "data_dependency": "no",
  "answerability": "faq"
}

RESPONSE:
1. A late payment fee, as mentioned in the Schedule of Charges, will be levied to the Card Account.
    2. The Bank is entitled to recover the outstanding dues in accordance with relevant laws.
    3. The Bank may appoint a third party for collection, at your cost and risk.
    4. You will be liable for all costs associated with the collection of dues and legal expenses with interest.

STABILITY SCORE: 1.0


In [23]:
# Example 4

account_id = "AXIS_CC_1006"
query = "There are transactions on my credit card that I did not authorize."

result_4 = process_query(
    query=query,
    account_id=account_id,
    index=index,
    embedding_model=embedding_model,
    rag_documents=rag_documents,
)

In [24]:
show_result(result_4)


DECISION:
{
  "query": "There are transactions on my credit card that I did not authorize.",
  "intent": "complaint",
  "issue": "account_issue",
  "risk_level": "high",
  "data_dependency": "NA",
  "answerability": "escalate"
}

RESPONSE:
Your concern requires additional review due to its sensitive nature. It has been flagged for appropriate handling to ensure it is addressed correctly.

STABILITY SCORE: 1.0
